# ทำความสะอาดข้อมูล ขั้นที่ 2/5: จัดการค่าว่าง (Missing Values)

| ขั้น | เรื่อง | สถานะ |
|---|---|---|
| 1 | ตรวจโครงสร้าง คีย์หลัก ข้อมูลซ้ำ ชนิดข้อมูล ขอบเขต | ✅ เสร็จแล้ว |
| **2 (ไฟล์นี้)** | **จัดการค่าว่าง** | เกณฑ์ 3.5.1.1.4, 3.5.1.2.1 |
| 3 | ค่าผิดปกติ (Outliers) | |
| 4 | มาตรฐานรูปแบบ + ความสอดคล้องของตัวเลข | |
| 5 | Data Dictionary + ไฟล์สะอาดสุดท้าย | |

**หลักการของขั้นนี้: ไม่เติมค่าว่างแบบเหมารวม แต่หาสาเหตุก่อน แล้วเลือกวิธีให้ตรงสาเหตุ**

| ลำดับ | ประเภทค่าว่าง | วิธีจัดการ |
|---|---|---|
| 2.2 | ค่าว่างที่ปลอมตัวเป็น 0 | เปลี่ยนเป็นค่าว่างก่อน (แทนที่ตามโดเมน) |
| 2.3 | ประเทศที่ไม่มีข้อมูลไฟฟ้าเลย | ลบประเทศนั้น |
| 2.4 | คำนวณได้จากคอลัมน์อื่น | คำนวณเติมด้วยสูตร (ตรวจสูตรก่อนใช้) |
| 2.5 | ขาดช่วงสั้น ๆ กลางอนุกรมเวลา | เติมเส้นตรงภายในประเทศเดียวกัน ไม่เกิน 2 ปี |
| 2.6 | แหล่งข้อมูลหยุดเก็บ (GDP หลังปี 2018) | ไม่เติม และระบุเป็นข้อจำกัด |

**อินพุต:** `clean_output/clean_step1.csv`  
**เอาต์พุต:** `clean_step2.csv`, `filled_cells.csv` (บันทึกทุกช่องที่ถูกแก้), `missing_before_after.csv` และเพิ่มบรรทัดใน `cleaning_log.csv`

> รันเรียงจากบนลงล่าง (Shift + Enter) หรือ **Kernel → Restart Kernel and Run All Cells** ต้องรันขั้นที่ 1 สำเร็จก่อน

## 2.1 โหลดผลจากขั้นที่ 1 และดูภาพรวมค่าว่าง

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

OUT_DIR = "clean_output"
IN_FILE = os.path.join(OUT_DIR, "clean_step1.csv")
LOG_FILE = os.path.join(OUT_DIR, "cleaning_log.csv")
if not os.path.exists(IN_FILE):
    raise FileNotFoundError("ไม่พบ clean_output/clean_step1.csv ให้รันไฟล์ clean_step1_structure.ipynb ให้สำเร็จก่อน")

df = pd.read_csv(IN_FILE)
df = df.sort_values(["country", "year"]).reset_index(drop=True)
before_all = df.copy()          # เก็บไว้เทียบ "ก่อน–หลัง" ตอนท้าย

# อ่าน log เดิมจากขั้นที่ 1 แล้วต่อท้าย (ถ้ารันซ้ำ จะลบบรรทัดของขั้นที่ 2 เดิมออกก่อน)
log_old = pd.read_csv(LOG_FILE)
log_old = log_old[~log_old["step"].str.startswith("2-")]
log_rows = []

def log_step(action, before, after, reason):
    log_rows.append({"step": "2-missing", "action": action,
                     "rows_before": before.shape[0], "rows_after": after.shape[0],
                     "cols_before": before.shape[1], "cols_after": after.shape[1], "reason": reason})
    print(f"✔ {action}: แถว {before.shape[0]:,} → {after.shape[0]:,} | คอลัมน์ {before.shape[1]} → {after.shape[1]}")

print("ข้อมูลจากขั้นที่ 1:", df.shape, "| ประเทศ:", df["country"].nunique())

In [ ]:
# ตารางเก็บว่าช่องไหนถูกเติมด้วยวิธีอะไร (ร่องรอยตรวจสอบย้อนกลับได้)
fill_method = pd.DataFrame("", index=df.index, columns=df.columns)

def fill(col, new_values, method, mask=None):
    """เติม col เฉพาะช่องที่ว่าง (และอยู่ใน mask ถ้ากำหนด) แล้วจดวิธีไว้ใน fill_method"""
    target = df[col].isna() & pd.Series(new_values, index=df.index).notna()
    if mask is not None:
        target &= mask
    df.loc[target, col] = pd.Series(new_values, index=df.index)[target]
    fill_method.loc[target, col] = method
    return int(target.sum())

def missing_table(data):
    return (data.isna().mean() * 100).round(2)

In [ ]:
miss_before = missing_table(df)
print("ค่าว่างรวมทั้งตาราง:", df.isna().sum().sum(), "ช่อง")
miss_before[miss_before > 0].sort_values().plot.barh(figsize=(8, 8), title="Missing % per column (before step 2)")
plt.axvline(5, color="red", ls="--"); plt.xlabel("% missing (red = 5%)"); plt.show()

In [ ]:
# missing map: แต่ละแถว = ประเทศ-ปี เรียงตามประเทศ, สีดำ = ว่าง
# ถ้าเห็นแถบดำยาวแนวนอนต่อเนื่อง = ประเทศนั้นไม่มีข้อมูลเกือบทุกคอลัมน์
def missing_map(data, title):
    plt.figure(figsize=(12, 4))
    plt.imshow(data.isna().values.T, aspect="auto", cmap="Greys", interpolation="nearest")
    plt.yticks(range(data.shape[1]), data.columns, fontsize=6)
    plt.xlabel("rows (country-year, sorted by country)"); plt.title(title); plt.show()

missing_map(df, "Missing map BEFORE step 2 (dark = missing)")

## 2.2 ค่าว่างที่ปลอมตัวเป็น 0 (Hidden missing)
ค่าว่างบางตัว **ไม่ได้แสดงเป็นช่องว่าง แต่ถูกใส่เป็น 0** ซึ่งอันตรายกว่า เพราะ `isna()` หาไม่เจอ และจะทำให้ค่าเฉลี่ยต่ำลงผิด ๆ

ใช้ **ความรู้เชิงโดเมน (domain rule)**: ประเทศที่มีประชากรอยู่จริง **ไม่มีทางผลิตไฟฟ้าได้ 0 TWh** เพราะทุกประเทศ/เขตปกครองมีโรงไฟฟ้า

In [ ]:
fake_zero = (df["electricity_generation"] == 0) & (df["population"] > 0)
print("แถวที่ผลิตไฟฟ้า = 0 แต่มีประชากร:", fake_zero.sum(), "แถว")
df.loc[fake_zero, "country"].value_counts()

> **อ่านผล:** Bermuda, Gibraltar, Niue, Saint Helena เป็น 0 ทุกปี ทั้งที่มีโรงไฟฟ้าจริง ส่วน Guadeloupe, Martinique, Reunion, French Guiana มีข้อมูลจริงถึงปี 2011 แล้วกลายเป็น 0 ตั้งแต่ปี 2012 (น่าจะเพราะแหล่งข้อมูลเลิกรายงานแยกจากฝรั่งเศส) และ Western Sahara เป็น 0 ตั้งแต่ปี 2010
> แปลว่า **0 ในที่นี้คือ "ไม่มีข้อมูล"** ไม่ใช่ "ผลิตได้ศูนย์" จึงเปลี่ยนเป็นค่าว่าง ทั้งคอลัมน์ไฟฟ้าทุกตัวของแถวนั้น

**ข้อระวัง:** 0 ในคอลัมน์เชื้อเพลิงย่อย เช่น `nuclear_electricity = 0` ของไทย **เป็นค่าจริง** (ไทยไม่มีโรงไฟฟ้านิวเคลียร์) ห้ามเปลี่ยน เราจึงเปลี่ยนเฉพาะแถวที่ไฟฟ้า "ทั้งประเทศ" เป็น 0

In [ ]:
ELEC_COLS = [c for c in df.columns
             if c.endswith("_electricity") or c.endswith("_share_elec") or c.endswith("_share_elec_exc_biofuel")
             or c in ["electricity_generation", "electricity_demand", "net_elec_imports",
                      "per_capita_electricity", "carbon_intensity_elec"]]

before = df.copy()
had_value = df[ELEC_COLS].notna() & fake_zero.values[:, None]
fill_method[ELEC_COLS] = fill_method[ELEC_COLS].mask(had_value, "zero_to_missing")
df.loc[fake_zero, ELEC_COLS] = np.nan
log_step("เปลี่ยน 0 ปลอมในคอลัมน์ไฟฟ้าเป็นค่าว่าง", before, df,
         f"{fake_zero.sum()} แถว ผลิตไฟฟ้า=0 ทั้งที่มีประชากร (domain rule)")

## 2.3 ลบประเทศที่ไม่มีข้อมูลไฟฟ้าเลย
ถ้าประเทศใดไม่มี `electricity_generation` เลยสักปี ก็ตอบคำถามเรื่องไฟฟ้าไม่ได้ และ **เติมไม่ได้** เพราะไม่มีค่าตั้งต้นของประเทศนั้นเอง

In [ ]:
no_elec = df.groupby("country")["electricity_generation"].apply(lambda s: s.isna().all())
drop_countries = no_elec[no_elec].index.tolist()
print(f"ประเทศที่ไม่มีข้อมูลไฟฟ้าเลย ({len(drop_countries)}):", drop_countries)

# เช็กให้แน่ใจว่าไม่ได้ลบประเทศในคำถามของเรา
ASEAN = ["Thailand", "Vietnam", "Malaysia", "Indonesia", "Philippines",
         "Singapore", "Myanmar", "Cambodia", "Laos", "Brunei"]
assert not set(drop_countries) & set(ASEAN), "มีประเทศอาเซียนถูกลบ ต้องตรวจสอบ!"
print("ประเทศอาเซียนไม่ถูกลบ ✅")

before = df.copy()
keep = ~df["country"].isin(drop_countries)
df = df[keep].copy(); fill_method = fill_method[keep].copy()
log_step("ลบประเทศที่ไม่มีข้อมูลไฟฟ้าเลยทุกปี", before, df,
         "ตอบคำถามไม่ได้และเติมไม่ได้: " + ", ".join(drop_countries))

## 2.4 คำนวณเติมจากคอลัมน์อื่น (Derive)
ข้อมูลไฟฟ้ามีความสัมพันธ์เชิงคณิตศาสตร์ เช่น
- ฟอสซิล = ถ่านหิน + ก๊าซ + น้ำมัน
- พลังงานหมุนเวียน = น้ำ + แสงอาทิตย์ + ลม + ชีวมวล + อื่น ๆ
- คาร์บอนต่ำ = พลังงานหมุนเวียน + นิวเคลียร์
- ความต้องการไฟฟ้า = ผลิตเอง + นำเข้าสุทธิ
- สัดส่วน (%) = ไฟฟ้าจากเชื้อเพลิงนั้น ÷ ไฟฟ้าทั้งหมด × 100
- ไฟฟ้าต่อคน (kWh) = ไฟฟ้าทั้งหมด (TWh) × 1,000,000,000 ÷ ประชากร (1 TWh = 10⁹ kWh)

ถ้าส่วนใดขาดไปตัวเดียว ก็คำนวณกลับได้ **วิธีนี้แม่นที่สุดเพราะไม่ได้เดา**

### ขั้นแรก: ตรวจสูตรก่อนใช้
ห้ามเชื่อสูตรเอง ต้องทดสอบกับแถวที่มีข้อมูลครบก่อนว่าสูตรให้ค่าตรงกับข้อมูลจริงหรือไม่

In [ ]:
FOSSIL = ["coal_electricity", "gas_electricity", "oil_electricity"]
RENEW = ["hydro_electricity", "solar_electricity", "wind_electricity",
         "biofuel_electricity", "other_renewable_exc_biofuel_electricity"]
IDENTITIES = {   # ผลรวม: [ส่วนประกอบ]
    "fossil_electricity": FOSSIL,
    "renewables_electricity": RENEW,
    "low_carbon_electricity": ["renewables_electricity", "nuclear_electricity"],
}

rows = []
for total, parts in IDENTITIES.items():
    ok = df[[total] + parts].notna().all(axis=1)
    err = (df.loc[ok, parts].sum(axis=1) - df.loc[ok, total]).abs()
    rows.append({"สูตร": f"{total} = " + " + ".join(parts), "แถวที่ทดสอบ": ok.sum(), "คลาดเคลื่อนสูงสุด (TWh)": err.max()})

ok = df[["electricity_demand", "electricity_generation", "net_elec_imports"]].notna().all(axis=1)
err = (df.loc[ok, "electricity_generation"] + df.loc[ok, "net_elec_imports"] - df.loc[ok, "electricity_demand"]).abs()
rows.append({"สูตร": "electricity_demand = generation + net_elec_imports", "แถวที่ทดสอบ": ok.sum(), "คลาดเคลื่อนสูงสุด (TWh)": err.max()})

ok = df[["per_capita_electricity", "electricity_generation", "population"]].notna().all(axis=1)
calc = df.loc[ok, "electricity_generation"] * 1e9 / df.loc[ok, "population"]
rows.append({"สูตร": "per_capita_electricity = generation x 1e9 / population",
             "แถวที่ทดสอบ": ok.sum(), "คลาดเคลื่อนสูงสุด (TWh)": None,
             "คลาดเคลื่อนสูงสุด (%)": ((calc / df.loc[ok, "per_capita_electricity"] - 1).abs() * 100).max()})

ok = df[["solar_share_elec", "solar_electricity", "electricity_generation"]].notna().all(axis=1) & (df["electricity_generation"] > 0)
calc = df.loc[ok, "solar_electricity"] / df.loc[ok, "electricity_generation"] * 100
rows.append({"สูตร": "solar_share_elec = solar_electricity / generation x 100",
             "แถวที่ทดสอบ": ok.sum(), "คลาดเคลื่อนสูงสุด (จุด %)": (calc - df.loc[ok, "solar_share_elec"]).abs().max()})
pd.DataFrame(rows)

> **อ่านผล:** ทุกสูตรคลาดเคลื่อนเกือบศูนย์ (ต่างแค่การปัดเศษ) จึงมั่นใจได้ว่าใช้สูตรเติมค่าว่างได้

In [ ]:
SHARE_MAP = {   # คอลัมน์สัดส่วน : คอลัมน์ปริมาณ
    "coal_share_elec": "coal_electricity", "gas_share_elec": "gas_electricity",
    "oil_share_elec": "oil_electricity", "nuclear_share_elec": "nuclear_electricity",
    "hydro_share_elec": "hydro_electricity", "solar_share_elec": "solar_electricity",
    "wind_share_elec": "wind_electricity", "biofuel_share_elec": "biofuel_electricity",
    "other_renewables_share_elec_exc_biofuel": "other_renewable_exc_biofuel_electricity",
    "fossil_share_elec": "fossil_electricity", "renewables_share_elec": "renewables_electricity",
    "low_carbon_share_elec": "low_carbon_electricity",
}

def derive_all():
    """เติมทุกช่องที่คำนวณได้จากสูตร คืนค่าจำนวนช่องที่เติม"""
    n = 0
    gen = df["electricity_generation"]
    # (ก) ส่วนประกอบที่ขาดตัวเดียว = ผลรวม - ส่วนที่เหลือ
    for total, parts in IDENTITIES.items():
        for p in parts:
            others = [x for x in parts if x != p]
            can = df[p].isna() & df[[total] + others].notna().all(axis=1)
            val = (df[total] - df[others].sum(axis=1)).clip(lower=0).round(4)
            n += fill(p, val, "derive_identity", can)
        # (ก2) ขาดหลายตัว แต่ส่วนที่รู้รวมกันเท่ากับผลรวมพอดี → ส่วนที่ขาดทั้งหมดต้องเป็น 0
        residual = df[total] - df[parts].sum(axis=1)          # sum ข้ามค่าว่าง
        zero_left = df[total].notna() & (residual.abs() < 1e-6)
        for p in parts:
            n += fill(p, pd.Series(0.0, index=df.index), "derive_identity_zero", zero_left)
        # (ข) ผลรวมที่ขาด = บวกส่วนประกอบ (เมื่อส่วนประกอบครบ)
        can = df[total].isna() & df[parts].notna().all(axis=1)
        n += fill(total, df[parts].sum(axis=1), "derive_identity", can)
    # (ค) ความต้องการไฟฟ้า / นำเข้าสุทธิ
    n += fill("electricity_demand", gen + df["net_elec_imports"], "derive_identity")
    n += fill("net_elec_imports", df["electricity_demand"] - gen, "derive_identity")
    # (ง) สัดส่วน % จากปริมาณ (เฉพาะเมื่อผลิตไฟฟ้า > 0 เพื่อเลี่ยงการหารด้วยศูนย์)
    pos = gen > 0
    for share, amount in SHARE_MAP.items():
        n += fill(share, df[amount] / gen * 100, "derive_formula", pos)
        n += fill(amount, df[share] * gen / 100, "derive_formula", pos)
    # (จ) ต่อหัว
    n += fill("per_capita_electricity", gen * 1e9 / df["population"], "derive_formula")
    n += fill("energy_per_capita", df["primary_energy_consumption"] * 1e9 / df["population"], "derive_formula")
    return n

before = df.copy()
total_derived = 0
for i in range(3):              # วนซ้ำ เพราะค่าที่เพิ่งเติม อาจทำให้คำนวณช่องอื่นต่อได้
    n = derive_all(); total_derived += n
    print(f"รอบที่ {i+1}: เติมได้ {n} ช่อง")
    if n == 0: break
log_step(f"คำนวณเติมจากสูตร ({total_derived} ช่อง)", before, df,
         "ใช้สูตรที่ตรวจแล้วว่าคลาดเคลื่อน ~0 เช่น ผลรวมเชื้อเพลิง, share = ส่วน/ทั้งหมด x 100")

> **ตัวอย่างที่เจอจริง:**
> - อินโดนีเซีย มาเลเซีย ไม่มีค่า `nuclear_electricity` แต่ `low_carbon = renewables` พอดี สูตรจึงได้ว่านิวเคลียร์ = 0 ซึ่งถูกต้อง เพราะสองประเทศนี้ไม่มีโรงไฟฟ้านิวเคลียร์
> - ถ้าขาดหลายตัว แต่ตัวที่รู้รวมกันได้เท่าผลรวมพอดี (เช่น ซาอุดีอาระเบีย ไม่มีค่าไฟฟ้าพลังน้ำ แต่แสงอาทิตย์ + ลม = พลังงานหมุนเวียนทั้งหมดแล้ว) ตัวที่ขาดต้องเป็น 0 ทั้งหมด (`derive_identity_zero`)

## 2.5 เติมช่วงขาดสั้น ๆ ด้วยเส้นตรง (Linear interpolation)
ถ้าประเทศหนึ่งมีค่าปี 2010 และ 2012 แต่ 2011 ว่าง ค่าที่สมเหตุสมผลที่สุดคือค่ากลางระหว่างสองปี

**กติกาที่ใช้ (เพื่อไม่ให้กลายเป็นการเดา):**
1. เติม **ภายในประเทศเดียวกัน** เท่านั้น (ห้ามเอาค่าประเทศอื่นหรือค่าเฉลี่ยโลกมาใส่)
2. เติมเฉพาะช่องว่าง **ตรงกลาง** ที่มีค่าทั้งก่อนและหลัง (ไม่เดาอนาคต ไม่เดาอดีต)
3. ช่วงว่างต้อง **ไม่เกิน 2 ปีติดกัน** ถ้ายาวกว่านั้นถือว่าไม่รู้จริง ปล่อยว่าง

เติมเฉพาะคอลัมน์ "ปริมาณ" แล้วค่อยใช้สูตรในข้อ 2.4 คำนวณสัดส่วนใหม่ ทำให้สัดส่วนยังรวมกันได้ 100% เสมอ

In [ ]:
MAX_GAP = 2
AMOUNT_COLS = ["population", "electricity_generation", "electricity_demand", "net_elec_imports",
               "carbon_intensity_elec", "greenhouse_gas_emissions", "primary_energy_consumption",
               "gdp"] + FOSSIL + RENEW + ["nuclear_electricity"]

def fill_short_gaps(s, max_gap=MAX_GAP):
    """s = อนุกรมของ 1 ประเทศ (index = ปี) เติมเฉพาะช่องว่างตรงกลางที่ยาว <= max_gap ปี"""
    interp = s.interpolate(method="index", limit_area="inside")
    is_na = s.isna()
    run_id = (~is_na).cumsum()                         # ช่องว่างที่ติดกันได้ id เดียวกัน
    run_len = is_na.groupby(run_id).transform("sum")
    return s.where(~(is_na & (run_len <= max_gap)), interp)

before = df.copy()
n_interp = 0
for col in AMOUNT_COLS:
    new = (df.set_index("year").groupby("country", observed=True)[col]
             .transform(fill_short_gaps).values)
    n_interp += fill(col, new, f"interpolate_le{MAX_GAP}y")
print("เติมด้วยเส้นตรงได้:", n_interp, "ช่อง")

if n_interp == 0:
    print("→ ไม่พบช่องว่างตรงกลางอนุกรมเวลาเลย ค่าว่างที่เหลืออยู่ต้นช่วงหรือท้ายช่วงทั้งหมด (ซึ่งเราไม่เติม)")
    print("  เก็บโค้ดนี้ไว้ เพราะถ้าได้ข้อมูลชุดใหม่ pipeline จะยังจัดการได้")
n2 = derive_all()   # คำนวณสัดส่วน/ต่อหัวใหม่จากค่าที่เพิ่งเติม
print("คำนวณสูตรต่อได้อีก:", n2, "ช่อง")
log_step(f"เติมช่วงขาดสั้น <= {MAX_GAP} ปี ภายในประเทศ ({n_interp} ช่อง) + คำนวณสูตรซ้ำ ({n2} ช่อง)",
         before, df, "ขาดช่วงสั้นกลางอนุกรมเวลา ค่ากลางระหว่างปีก่อน-หลังสมเหตุสมผล")

## 2.6 GDP หลังปี 2018: ไม่เติม และติดป้ายไว้
แหล่งข้อมูล GDP ในไฟล์นี้ **สิ้นสุดที่ปี 2018 ทุกประเทศ** ถ้าเติมปี 2019–2022 ก็คือการเดาอนาคต (และปี 2020 มีโควิดที่ GDP ตกแรง เส้นตรงจะผิดแน่นอน)

**ตัดสินใจ:** ไม่เติม และเพิ่มคอลัมน์ `gdp_available` ให้แดชบอร์ดรู้ว่าปีไหนใช้ GDP ได้

In [ ]:
last_gdp_year = df.loc[df["gdp"].notna(), "year"].max()
print("ปีสุดท้ายที่มี GDP:", last_gdp_year)
print("จำนวนแถว GDP ว่างหลังปีนี้:", (df["year"] > last_gdp_year).sum())

before = df.copy()
df["gdp_available"] = df["gdp"].notna()
fill_method["gdp_available"] = ""
log_step("เพิ่มคอลัมน์ gdp_available (ไม่เติม GDP หลังปี 2018)", before, df,
         f"แหล่งข้อมูล GDP สิ้นสุดปี {last_gdp_year}; เติมต่อ = เดาอนาคต")

## 2.7 ค่าว่างที่ยังเหลือ และเหตุผล
ค่าว่างที่เหลือคือสิ่งที่ **ไม่มีหลักฐานพอให้เติม** ซึ่งการปล่อยไว้แล้วบอกเหตุผลถูกต้องกว่าการเติมมั่ว

In [ ]:
remain = df.isna().sum()
remain = remain[remain > 0].sort_values(ascending=False)
remain_tbl = pd.DataFrame({"ว่าง (ช่อง)": remain, "ว่าง (%)": (remain / len(df) * 100).round(2)})
# ดูว่าค่าว่างที่เหลือกระจุกอยู่ประเทศ/ปีไหน
remain_tbl["ประเทศที่ว่างมากสุด"] = [
    ", ".join(df.loc[df[c].isna(), "country"].value_counts().head(3).index) for c in remain.index]
remain_tbl["ว่างในปี 2022 (%)"] = [
    round(df.loc[df[c].isna(), "year"].eq(2022).mean() * 100) for c in remain.index]
remain_tbl

> **อ่านผล:** นอกจาก GDP แล้ว ค่าว่างที่เหลือกระจุกอยู่ที่ **เขตปกครองเล็ก ๆ** (เช่น Western Sahara, French Guiana) ในปีที่แหล่งข้อมูลยังไม่เริ่มเก็บ และส่วนหนึ่งอยู่ใน **ปี 2022** ที่ข้อมูลยังไม่ครบ
> ทั้งหมดไม่กระทบประเทศในคำถามของเรา (ดูผลของไทยในข้อ 2.9) และเราจะติดป้ายปี 2022 ว่าข้อมูลไม่ครบในขั้นที่ 4

## 2.8 หลักฐานก่อน–หลัง

In [ ]:
missing_map(df.drop(columns="gdp_available"), "Missing map AFTER step 2 (dark = missing)")

miss_after = missing_table(df.drop(columns="gdp_available"))
compare = pd.DataFrame({"ว่างก่อน (%)": miss_before, "ว่างหลัง (%)": miss_after})
compare["ลดลง (จุด %)"] = (compare["ว่างก่อน (%)"] - compare["ว่างหลัง (%)"]).round(2)
compare = compare.sort_values("ว่างก่อน (%)", ascending=False)

ax = compare[["ว่างก่อน (%)", "ว่างหลัง (%)"]].rename(columns={"ว่างก่อน (%)": "before", "ว่างหลัง (%)": "after"}) \
        .drop(index=["gdp"]).plot.barh(figsize=(8, 9), title="Missing % per column: before vs after step 2")
ax.axvline(5, color="red", ls="--"); ax.invert_yaxis(); ax.set_xlabel("% missing (red = 5%)"); plt.show()
compare

### ผลกระทบต่อสถิติ
เกณฑ์ 3.5.1.2.1 ต้องการให้แสดงว่าการเติมค่า **ไม่ทำให้สถิติเพี้ยน** จึงเทียบค่าเฉลี่ยและมัธยฐานก่อน–หลัง
ถ้าเปลี่ยนไปไม่มาก แปลว่าการเติมสมเหตุสมผล

In [ ]:
KEY_STATS = ["electricity_generation", "per_capita_electricity", "renewables_share_elec",
             "fossil_share_elec", "gas_share_elec", "solar_share_elec", "carbon_intensity_elec", "population"]
b = before_all[KEY_STATS]
a = df[KEY_STATS]
stats = pd.DataFrame({
    "n ก่อน": b.count(), "n หลัง": a.count(),
    "mean ก่อน": b.mean().round(2), "mean หลัง": a.mean().round(2),
    "median ก่อน": b.median().round(2), "median หลัง": a.median().round(2),
})
stats["Δ mean (%)"] = ((stats["mean หลัง"] / stats["mean ก่อน"] - 1) * 100).round(2)
stats

> **อ่านผล:** ค่า `n` เพิ่มขึ้น (มีข้อมูลใช้มากขึ้น) ขณะที่ mean และ median เปลี่ยนไปเล็กน้อย
> ข้อสังเกต: `electricity_generation`, `per_capita_electricity`, `population` ขยับขึ้นราว 3% เพราะเรา **ลบ 0 ปลอม และลบเขตปกครองที่ไม่มีข้อมูลไฟฟ้า** ออก (0 ปลอมเคยดึงค่าเฉลี่ยให้ต่ำผิดจริง) จึงเป็นการแก้ให้ถูกขึ้น ไม่ใช่ทำให้เพี้ยน ส่วนคอลัมน์สัดส่วนที่ถูกคำนวณเติม ค่าเฉลี่ยแทบไม่เปลี่ยนเลย

In [ ]:
# สรุปว่าเติมด้วยวิธีไหนไปกี่ช่อง
methods = fill_method.stack()
methods = methods[methods != ""]
print("จำนวนช่องที่ถูกแก้ แยกตามวิธี:")
methods.value_counts().to_frame("ช่อง")

## 2.9 ตรวจเกณฑ์ค่าว่าง (3.5.1.1.4)
- คีย์ (country, iso_code, year) ต้องว่าง **≤ 1%**
- ฟีเจอร์สำคัญที่คำถามใช้ ต้องว่าง **≤ 5%**

In [ ]:
KEY = ["country", "iso_code", "year"]
IMPORTANT = ["population", "electricity_generation", "electricity_demand", "per_capita_electricity",
             "renewables_share_elec", "fossil_share_elec", "gas_share_elec", "coal_share_elec",
             "hydro_share_elec", "solar_share_elec", "wind_share_elec", "carbon_intensity_elec"]

check = pd.concat([
    pd.DataFrame({"ประเภท": "คีย์", "ว่าง (%)": missing_table(df[KEY]), "เกณฑ์ (%)": 1}),
    pd.DataFrame({"ประเภท": "ฟีเจอร์สำคัญ", "ว่าง (%)": missing_table(df[IMPORTANT]), "เกณฑ์ (%)": 5}),
])
check["ผ่าน"] = np.where(check["ว่าง (%)"] <= check["เกณฑ์ (%)"], "✅", "❌")
check

In [ ]:
# ประเทศไทย: เหลือว่างอะไรบ้าง
th = df[df["country"] == "Thailand"]
th_missing = th.isna().sum()
print("ประเทศไทย", len(th), "ปี | คอลัมน์ที่ยังว่าง:")
print(th_missing[th_missing > 0].to_string() if th_missing.any() else "ไม่มี")

## 2.10 บันทึกไฟล์

In [ ]:
df.to_csv(os.path.join(OUT_DIR, "clean_step2.csv"), index=False, encoding="utf-8-sig")

# filled_cells.csv: ทุกช่องที่ถูกแก้ = ประเทศ, ปี, คอลัมน์, วิธี, ค่าเดิม, ค่าใหม่
audit = methods.reset_index(); audit.columns = ["row", "column", "method"]
audit["country"] = df.loc[audit["row"], "country"].values
audit["year"] = df.loc[audit["row"], "year"].values
audit["old_value"] = [before_all.at[r, c] for r, c in zip(audit["row"], audit["column"])]
audit["new_value"] = [df.at[r, c] for r, c in zip(audit["row"], audit["column"])]
audit = audit[["country", "year", "column", "method", "old_value", "new_value"]]
audit.to_csv(os.path.join(OUT_DIR, "filled_cells.csv"), index=False, encoding="utf-8-sig")

compare.to_csv(os.path.join(OUT_DIR, "missing_before_after.csv"), encoding="utf-8-sig")
pd.concat([log_old, pd.DataFrame(log_rows)]).to_csv(LOG_FILE, index=False, encoding="utf-8-sig")

print("บันทึกแล้ว:")
for f in sorted(os.listdir(OUT_DIR)):
    print("  -", f, f"({os.path.getsize(os.path.join(OUT_DIR, f))/1024:,.0f} KB)")
print("\nตัวอย่าง filled_cells.csv (วิธีละ 3 แถว):")
audit.groupby("method").head(3)

In [ ]:
pd.read_csv(LOG_FILE)

## ✅ เช็กลิสต์ก่อนไปขั้นที่ 3
- [ ] รันครบไม่มี error และตาราง 2.9 ผ่าน ✅ ทุกแถว
- [ ] ในโฟลเดอร์ `clean_output/` มีไฟล์ใหม่ `clean_step2.csv`, `filled_cells.csv`, `missing_before_after.csv`
- [ ] เปิด `filled_cells.csv` ใน Excel แล้วลองกรองดูว่าเติมอะไรไปบ้าง
- [ ] อธิบายได้ว่า **ทำไม 0 ของ Bermuda ถึงเป็นค่าว่าง แต่ 0 นิวเคลียร์ของไทยเป็นค่าจริง**
- [ ] อธิบายได้ว่า **ทำไมไม่เติม GDP** และ **ทำไมไม่เติมด้วยค่าเฉลี่ยโลก**

**ประโยคสรุปไว้ตอบกรรมการ:** "เราไม่เติมค่าว่างแบบเหมารวม แต่แยกตามสาเหตุ: 0 ปลอมเปลี่ยนเป็นค่าว่าง, ไม่มีข้อมูลเลยลบออก, มีสูตรก็คำนวณ, ขาดช่วงสั้นเติมเส้นตรงภายในประเทศ, แหล่งหยุดเก็บไม่เติม แล้วแสดงก่อน–หลังว่าสถิติไม่เพี้ยน ทุกช่องที่แก้ตรวจย้อนได้ในไฟล์ filled_cells.csv"

**ขั้นถัดไป (ไฟล์ที่ 3):** หาค่าผิดปกติ (Outliers) ด้วย IQR และ Z-score แยกให้ได้ว่าค่าไหน "ผิดจริง" และค่าไหน "สูงจริง" (เช่น ไอซ์แลนด์ใช้ไฟต่อคนสูงที่สุดในโลกเพราะมีโรงถลุงอะลูมิเนียม)